# Clase 02 — Laboratorio

**Estudiante:** Sebastián Cruz Cardona  
**Código:** 240220222010  
**Dominio independiente:** Asistente normativo  
**Propósito:** mostrar un método breve para comparar configuraciones.


*Nota: los pasos no necesarios para el objetivo del laboratorio fueron omitidos.*

In [1]:
# 0. Preparación del entorno (ejecutar antes de la clase)
%pip install -q openai tiktoken pandas matplotlib ollama

import math
import os
import time
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
import tiktoken
from openai import OpenAI

pd.set_option("display.max_colwidth", 100)
print("Entorno preparado.")

Entorno preparado.


## Construir un contexto mínimo

In [21]:
# Paso 4: contexto mínimo, una pregunta y conteo previo
instruccion = """Eres el asistente normativo constitucional de un juzgado en
Colombia.
Responde únicamente con base en los artículos de la Constitución Política de
Colombia proporcionados. Cuando respondas, incluye el artículo en el que se basa
la respuesta.
Si falta información, dilo claramente. Responde en máximo 45 palabras."""

constitucion = """CONSTITUCIÓN POLÍTICA DE COLOMBIA
ARTICULO 1o. Colombia es un Estado social de derecho, organizado en forma de
República unitaria, descentralizada, con autonomía de sus entidades
territoriales, democrática, participativa y pluralista,
fundada en el respeto de la dignidad humana, en el trabajo y la solidaridad
de las personas que la integran y en la prevalencia del interés general."""

# Preguntas (normal y ambigua)
pregunta = "En Colombia prevalece el interés particular sobre el general o al contrario?"
pregunta_ambigua = "Cuáles son los derechos fundamentales en Colombia"

messages = [
    {"role": "system", "content": instruccion},
    {"role": "user", "content": f"{constitucion}\n\nPREGUNTA: {pregunta}"},
]

messages_ambiguo = [
    {"role": "system", "content": instruccion},
    {"role": "user", "content": f"{constitucion}\n\nPREGUNTA: {pregunta_ambigua}"},
]

texto_contexto = "\n".join(mensaje["content"] for mensaje in messages)
tokens_contexto_estimados = len(encoding.encode(texto_contexto))

print(texto_contexto)
print(f"\nEstimación de referencia (tokenizador de OpenAI): {tokens_contexto_estimados} tokens")
print("El conteo real del modelo local aparecerá en la columna tokens_entrada del paso 5.")

Eres el asistente normativo constitucional de un juzgado en 
Colombia.
Responde únicamente con base en los artículos de la Constitución Política de 
Colombia proporcionados. Cuando respondas, incluye el artículo en el que se basa
la respuesta.
Si falta información, dilo claramente. Responde en máximo 45 palabras.
CONSTITUCIÓN POLÍTICA DE COLOMBIA
ARTICULO 1o. Colombia es un Estado social de derecho, organizado en forma de 
República unitaria, descentralizada, con autonomía de sus entidades 
territoriales, democrática, participativa y pluralista, 
fundada en el respeto de la dignidad humana, en el trabajo y la solidaridad 
de las personas que la integran y en la prevalencia del interés general.

PREGUNTA: En Colombia prevalece el interés particular sobre el general o al contrario?

Estimación de referencia (tokenizador de OpenAI): 181 tokens
El conteo real del modelo local aparecerá en la columna tokens_entrada del paso 5.


## Conectar modelo


In [5]:
!apt-get install -y zstd
!zstd --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (303 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
*** zstd command line interface 64-bits v1.4.8, by Yann Collet ***


In [6]:
# Instalar Ollama en la sesión de Colab
!sudo apt-get install -y -q pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# La instalación debe reportar "NVIDIA GPU installed". Si no aparece, revise el entorno de ejecución.
!nvidia-smi --query-gpu=name,memory.total --format=csv

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 57 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 2s (210 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initial

In [7]:
import subprocess
import time
import ollama

In [8]:
# Iniciar el servidor local y descargar el modelo
# Verifique el nombre en https://ollama.com/library antes de clase.
MODEL_NAME = "gemma4:e2b"

subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull {MODEL_NAME}

In [9]:
# Respuesta de prueba: carga el modelo en memoria y confirma que responde.
prueba = ollama.chat(
    model=MODEL_NAME,
    messages=messages,
    think=False,
    options={"temperature": 0.2, "num_predict": 300},
)
print(prueba["message"]["content"])
print("thinking:", prueba["message"].get("thinking"))

Prevalece el interés general. (Artículo 1°)
thinking: None


In [10]:
# El modelo debe aparecer con PROCESSOR = 100% GPU.
!ollama ps

NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
gemma4:e2b    7fbdbf8f5e45    1.7 GB    100% GPU     4096       4 minutes from now    


## Ejecutar tres configuraciones


### 5.1 Pegunta correcta

In [23]:
# Pregunta correcta con número de predicciones = 300
# Paso 5 y 6: generar con tres temperaturas y registrar tokens + latencia
NANOSEGUNDOS = 1_000_000_000

def ejecutar_prueba(temperatura, top_p=1.0, num_predict=300):
    inicio = time.perf_counter()
    respuesta = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={
            "temperature": temperatura,
            "top_p": top_p,
            "num_predict": num_predict,
        },
    )
    latencia = time.perf_counter() - inicio

    tokens_entrada = respuesta.get("prompt_eval_count")
    tokens_salida = respuesta.get("eval_count")
    carga = respuesta.get("load_duration", 0) / NANOSEGUNDOS

    return {
        "temperatura": temperatura,
        "top_p": top_p,
        "num_predict": num_predict,
        "tokens_entrada": tokens_entrada,
        "tokens_salida": tokens_salida,
        "tokens_total": (tokens_entrada or 0) + (tokens_salida or 0),
        "latencia_s": round(latencia, 2),
        "carga_s": round(carga, 2),
        "respuesta": respuesta["message"]["content"].strip(),
    }

resultados = []
for temperatura in [0.0, 0.7, 1.2]:
    print(f"Ejecutando temperatura={temperatura}...")
    resultados.append(ejecutar_prueba(temperatura=temperatura))

resultados_df = pd.DataFrame(resultados)
display(resultados_df)

print(
    f"Tokens de entrada estimados con tiktoken: {tokens_contexto_estimados} | "
    f"reportados por {MODEL_NAME}: {resultados_df['tokens_entrada'].iloc[0]}"
)

Ejecutando temperatura=0.0...
Ejecutando temperatura=0.7...
Ejecutando temperatura=1.2...


,temperatura,top_p,num_predict,tokens_entrada,tokens_salida,tokens_total,latencia_s,carga_s,respuesta
0,0.0,1.0,300,202,300,502,37.48,33.69,
1,0.7,1.0,300,202,300,502,3.47,0.00,
2,1.2,1.0,300,202,296,498,3.44,0.00,Prevalece el interés general. Esto se establece en la Constitución Política de Colombia.\n\nArtí...


Tokens de entrada estimados con tiktoken: 181 | reportados por gemma4:e2b: 202


In [24]:
for r in resultados:
    print(r["temperatura"], "->", repr(r["respuesta"])[:300])

0.0 -> ''
0.7 -> ''
1.2 -> 'Prevalece el interés general. Esto se establece en la Constitución Política de Colombia.\n\nArtículo 1º.'


In [26]:
# Resumen compacto para proyectar y copiar a la bitácora
resumen = resultados_df[
    ["temperatura", "tokens_entrada", "tokens_salida", "latencia_s", "carga_s", "respuesta"]
].copy()
resumen["cumple_datos"] = "REVISAR"
resumen["decision"] = "CONSERVAR / DESCARTAR"

display(resumen)

print("\nDECISIÓN MODELO PARA LA BITÁCORA")
print("Conservamos temperatura = Todavía no se toma una decisión, se requiere obtener las 3 respuestas.")
print("Observamos como limitación el número de predicciones, que no permite la generación de textos más largos.")
print("En la siguiente prueba cambiaremos únicamente el número de predicciones.")

,temperatura,tokens_entrada,tokens_salida,latencia_s,carga_s,respuesta,cumple_datos,decision
0,0.0,202,300,37.48,33.69,,REVISAR,CONSERVAR / DESCARTAR
1,0.7,202,300,3.47,0.00,,REVISAR,CONSERVAR / DESCARTAR
2,1.2,202,296,3.44,0.00,Prevalece el interés general. Esto se establece en la Constitución Política de Colombia.\n\nArtí...,REVISAR,CONSERVAR / DESCARTAR



DECISIÓN MODELO PARA LA BITÁCORA
Conservamos temperatura = Todavía no se toma una decisión, se requiere obtener las 3 respuestas.
Observamos como limitación el número de predicciones, que no permite la generación de textos más largos.
En la siguiente prueba cambiaremos únicamente el número de predicciones.


In [14]:
# Pregunta correcta con número de predicciones = 400
# Paso 5 y 6: generar con tres temperaturas y registrar tokens + latencia
NANOSEGUNDOS = 1_000_000_000

def ejecutar_prueba(temperatura, top_p=1.0, num_predict=400):
    inicio = time.perf_counter()
    respuesta = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={
            "temperature": temperatura,
            "top_p": top_p,
            "num_predict": num_predict,
        },
    )
    latencia = time.perf_counter() - inicio

    tokens_entrada = respuesta.get("prompt_eval_count")
    tokens_salida = respuesta.get("eval_count")
    carga = respuesta.get("load_duration", 0) / NANOSEGUNDOS

    return {
        "temperatura": temperatura,
        "top_p": top_p,
        "num_predict": num_predict,
        "tokens_entrada": tokens_entrada,
        "tokens_salida": tokens_salida,
        "tokens_total": (tokens_entrada or 0) + (tokens_salida or 0),
        "latencia_s": round(latencia, 2),
        "carga_s": round(carga, 2),
        "respuesta": respuesta["message"]["content"].strip(),
    }

resultados = []
for temperatura in [0.0, 0.7, 1.2]:
    print(f"Ejecutando temperatura={temperatura}...")
    resultados.append(ejecutar_prueba(temperatura=temperatura))

resultados_df = pd.DataFrame(resultados)
display(resultados_df)

print(
    f"Tokens de entrada estimados con tiktoken: {tokens_contexto_estimados} | "
    f"reportados por {MODEL_NAME}: {resultados_df['tokens_entrada'].iloc[0]}"
)

Ejecutando temperatura=0.0...
Ejecutando temperatura=0.7...
Ejecutando temperatura=1.2...


,temperatura,top_p,num_predict,tokens_entrada,tokens_salida,tokens_total,latencia_s,carga_s,respuesta
0,0.0,1.0,400,202,305,507,4.12,0.0,Prevalece el interés general. Esto se establece como fundamento de la organización del Estado. (...
1,0.7,1.0,400,202,326,528,4.56,0.0,Prevalece el interés general. La Constitución establece que la República se funda en la prevalen...
2,1.2,1.0,400,202,331,533,3.94,0.0,"Prevalece el interés general. Esto se fundamenta en la mención explícita de la ""prevalencia del ..."


Tokens de entrada estimados con tiktoken: 181 | reportados por gemma4:e2b: 202


In [16]:
for r in resultados:
    print(r["temperatura"], "->", repr(r["respuesta"])[:300])

0.0 -> 'Prevalece el interés general. Esto se establece como fundamento de la organización del Estado. (Artículo 1)'
0.7 -> 'Prevalece el interés general. La Constitución establece que la República se funda en la prevalencia del interés general. (Artículo 1°)'
1.2 -> 'Prevalece el interés general. Esto se fundamenta en la mención explícita de la "prevalencia del interés general". (Artículo 1o).'


In [22]:
# Resumen compacto para proyectar y copiar a la bitácora
resumen = resultados_df[
    ["temperatura", "tokens_entrada", "tokens_salida", "latencia_s", "carga_s", "respuesta"]
].copy()
resumen["cumple_datos"] = "REVISAR"
resumen["decision"] = "CONSERVAR / DESCARTAR"

display(resumen)

print("\nDECISIÓN MODELO PARA LA BITÁCORA")
print("Conservamos temperatura = 0.0 porque es la que menos tokens de salida consume y es equilibrada en cuanto a la latencia. "
"Por otra parte, la respuesta es la que mejor información contextual brinda con respecto a la información de base, mientras "
"se mantiene concisa.")
print("Observamos como limitación n/a.")
print("En la siguiente prueba cambiaremos únicamente n/a.")

,temperatura,tokens_entrada,tokens_salida,latencia_s,carga_s,respuesta,cumple_datos,decision
0,0.0,202,305,4.12,0.0,Prevalece el interés general. Esto se establece como fundamento de la organización del Estado. (...,REVISAR,CONSERVAR / DESCARTAR
1,0.7,202,326,4.56,0.0,Prevalece el interés general. La Constitución establece que la República se funda en la prevalen...,REVISAR,CONSERVAR / DESCARTAR
2,1.2,202,331,3.94,0.0,"Prevalece el interés general. Esto se fundamenta en la mención explícita de la ""prevalencia del ...",REVISAR,CONSERVAR / DESCARTAR



DECISIÓN MODELO PARA LA BITÁCORA
Conservamos temperatura = 0.0 porque es la que menos tokens de salida consume y es equilibrada en cuanto a la latencia. Por otra parte, la respuesta es la que mejor información contextual brinda con respecto a la información de base, mientras se mantiene concisa.
Observamos como limitación n/a.
En la siguiente prueba cambiaremos únicamente n/a.


### 5.2 Pegunta incorrecta o ambigua

In [38]:
# Pregunta incorrecta o ambigua con 400 predicciones
# Paso 5 y 6: generar con tres temperaturas y registrar tokens + latencia
NANOSEGUNDOS = 1_000_000_000

def ejecutar_prueba(temperatura, top_p=1.0, num_predict=400):
    inicio = time.perf_counter()
    respuesta = ollama.chat(
        model=MODEL_NAME,
        messages=messages_ambiguo,
        options={
            "temperature": temperatura,
            "top_p": top_p,
            "num_predict": num_predict,
        },
    )
    latencia = time.perf_counter() - inicio

    tokens_entrada = respuesta.get("prompt_eval_count")
    tokens_salida = respuesta.get("eval_count")
    carga = respuesta.get("load_duration", 0) / NANOSEGUNDOS

    return {
        "temperatura": temperatura,
        "top_p": top_p,
        "num_predict": num_predict,
        "tokens_entrada": tokens_entrada,
        "tokens_salida": tokens_salida,
        "tokens_total": (tokens_entrada or 0) + (tokens_salida or 0),
        "latencia_s": round(latencia, 2),
        "carga_s": round(carga, 2),
        "respuesta": respuesta["message"]["content"].strip(),
    }

resultados = []
for temperatura in [0.0, 0.7, 1.2]:
    print(f"Ejecutando temperatura={temperatura}...")
    resultados.append(ejecutar_prueba(temperatura=temperatura))

resultados_df = pd.DataFrame(resultados)
display(resultados_df)

print(
    f"Tokens de entrada estimados con tiktoken: {tokens_contexto_estimados} | "
    f"reportados por {MODEL_NAME}: {resultados_df['tokens_entrada'].iloc[0]}"
)

Ejecutando temperatura=0.0...
Ejecutando temperatura=0.7...
Ejecutando temperatura=1.2...


,temperatura,top_p,num_predict,tokens_entrada,tokens_salida,tokens_total,latencia_s,carga_s,respuesta
0,0.0,1.0,400,196,400,596,5.11,0.0,
1,0.7,1.0,400,196,370,566,4.93,0.0,La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporc...
2,1.2,1.0,400,196,400,596,4.83,0.0,


Tokens de entrada estimados con tiktoken: 181 | reportados por gemma4:e2b: 196


In [39]:
for r in resultados:
    print(r["temperatura"], "->", repr(r["respuesta"])[:300])

0.0 -> ''
0.7 -> 'La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporcionado.'
1.2 -> ''


In [40]:
# Resumen compacto para proyectar y copiar a la bitácora
resumen = resultados_df[
    ["temperatura", "tokens_entrada", "tokens_salida", "latencia_s", "carga_s", "respuesta"]
].copy()
resumen["cumple_datos"] = "REVISAR"
resumen["decision"] = "CONSERVAR / DESCARTAR"

display(resumen)

print("\nDECISIÓN MODELO PARA LA BITÁCORA")
print("Conservamos temperatura = n/a")
print("Observamos como limitación nuevamente el número de predicciones no alcanza para la generación completa de las respuestas")
print("En la siguiente prueba cambiaremos únicamente el número de predicciones")

,temperatura,tokens_entrada,tokens_salida,latencia_s,carga_s,respuesta,cumple_datos,decision
0,0.0,196,400,5.11,0.0,,REVISAR,CONSERVAR / DESCARTAR
1,0.7,196,370,4.93,0.0,La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporc...,REVISAR,CONSERVAR / DESCARTAR
2,1.2,196,400,4.83,0.0,,REVISAR,CONSERVAR / DESCARTAR



DECISIÓN MODELO PARA LA BITÁCORA
Conservamos temperatura = n/a
Observamos como limitación nuevamente el número de predicciones no alcanza para la generación completa de las respuestas
En la siguiente prueba cambiaremos únicamente el número de predicciones


In [29]:
# Pregunta incorrecta o ambigua con 500 predicciones
# Paso 5 y 6: generar con tres temperaturas y registrar tokens + latencia
NANOSEGUNDOS = 1_000_000_000

def ejecutar_prueba(temperatura, top_p=1.0, num_predict=500):
    inicio = time.perf_counter()
    respuesta = ollama.chat(
        model=MODEL_NAME,
        messages=messages_ambiguo,
        options={
            "temperature": temperatura,
            "top_p": top_p,
            "num_predict": num_predict,
        },
    )
    latencia = time.perf_counter() - inicio

    tokens_entrada = respuesta.get("prompt_eval_count")
    tokens_salida = respuesta.get("eval_count")
    carga = respuesta.get("load_duration", 0) / NANOSEGUNDOS

    return {
        "temperatura": temperatura,
        "top_p": top_p,
        "num_predict": num_predict,
        "tokens_entrada": tokens_entrada,
        "tokens_salida": tokens_salida,
        "tokens_total": (tokens_entrada or 0) + (tokens_salida or 0),
        "latencia_s": round(latencia, 2),
        "carga_s": round(carga, 2),
        "respuesta": respuesta["message"]["content"].strip(),
    }

resultados = []
for temperatura in [0.0, 0.7, 1.2]:
    print(f"Ejecutando temperatura={temperatura}...")
    resultados.append(ejecutar_prueba(temperatura=temperatura))

resultados_df = pd.DataFrame(resultados)
display(resultados_df)

print(
    f"Tokens de entrada estimados con tiktoken: {tokens_contexto_estimados} | "
    f"reportados por {MODEL_NAME}: {resultados_df['tokens_entrada'].iloc[0]}"
)

Ejecutando temperatura=0.0...
Ejecutando temperatura=0.7...
Ejecutando temperatura=1.2...


,temperatura,top_p,num_predict,tokens_entrada,tokens_salida,tokens_total,latencia_s,carga_s,respuesta
0,0.0,1.0,500,196,434,630,5.17,0.0,La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporc...
1,0.7,1.0,500,196,475,671,5.62,0.0,La información proporcionada no enumera los derechos fundamentales. Solo establece que Colombia ...
2,1.2,1.0,500,196,419,615,4.96,0.0,La Constitución Política de Colombia no detalla cuáles son los derechos fundamentales en el artí...


Tokens de entrada estimados con tiktoken: 181 | reportados por gemma4:e2b: 196


In [30]:
for r in resultados:
    print(r["temperatura"], "->", repr(r["respuesta"])[:300])

0.0 -> 'La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporcionado.'
0.7 -> 'La información proporcionada no enumera los derechos fundamentales. Solo establece que Colombia es un Estado social de derecho fundado en el respeto de la dignidad humana. (Artículo 1)'
1.2 -> 'La Constitución Política de Colombia no detalla cuáles son los derechos fundamentales en el artículo proporcionado. El artículo 1 establece los fundamentos del Estado, como la dignidad humana.'


In [31]:
# Resumen compacto para proyectar y copiar a la bitácora
resumen = resultados_df[
    ["temperatura", "tokens_entrada", "tokens_salida", "latencia_s", "carga_s", "respuesta"]
].copy()
resumen["cumple_datos"] = "REVISAR"
resumen["decision"] = "CONSERVAR / DESCARTAR"

display(resumen)

print("\nDECISIÓN MODELO PARA LA BITÁCORA")
print("Conservamos temperatura = 0.0 porque es equilibrada (intermedia) en cuanto a tokens de salida y latencia, mientras "
" la respuesta ante una pregunta que no puede contestar se manitne concisa y no gasta tokens de manera inecesaria.")
print("Observamos como limitación n/a.")
print("En la siguiente prueba cambiaremos únicamente n/a.")

,temperatura,tokens_entrada,tokens_salida,latencia_s,carga_s,respuesta,cumple_datos,decision
0,0.0,196,434,5.17,0.0,La información sobre los derechos fundamentales no se encuentra detallada en el artículo proporc...,REVISAR,CONSERVAR / DESCARTAR
1,0.7,196,475,5.62,0.0,La información proporcionada no enumera los derechos fundamentales. Solo establece que Colombia ...,REVISAR,CONSERVAR / DESCARTAR
2,1.2,196,419,4.96,0.0,La Constitución Política de Colombia no detalla cuáles son los derechos fundamentales en el artí...,REVISAR,CONSERVAR / DESCARTAR



DECISIÓN MODELO PARA LA BITÁCORA
Conservamos temperatura = 0.0 porque es la que menos tokens de salida consume y es equilibrada en cuanto a la latencia. Por otra parte, la respuesta es la que mejor información contextual brinda con respecto a la información de base, mientras se mantiene concisa.
Observamos como limitación n/a.
En la siguiente prueba cambiaremos únicamente n/a.
